# 🌍 AI Travel Agent

An intelligent travel planning agent powered by **LangGraph** and **Groq LLM**.  
Give it a plain-English request and it will:

1. **Route** — detect if query is travel-related or not  
2. **Parse** your request into structured travel parameters  
3. **Fetch real-time data** — weather, currency rates, hotels, attractions (in parallel)  
4. **Calculate** budget breakdown with currency conversions  
5. **Compose** a day-by-day itinerary  
6. **Summarize & Export** the plan as a clean Markdown file  

### Architecture
```
__start__
   │
   ├── TRAVEL ──→ query_analyzer ──→ geocode
   │                                   │
   │                    ┌───────────────┼───────────────┐
   │                    ▼               ▼               ▼
   │              hotel_agent    weather_agent    attractions_agent
   │                    │               │               │
   │                    └───────────────┼───────────────┘
   │                                   ▼
   │                           calculator_agent
   │                                   ▼
   │                          itinerary_agent
   │                                   ▼
   │                           summary_agent ──→ __end__
   │
   └── NOT_TRAVEL ──→ __end__
```

### APIs Used
| Service | Provider | Cost |
|---------|----------|------|
| LLM | Groq (Llama 3.3 70B) | Free tier |
| Weather | Open-Meteo | Free, no key |
| Currency | Frankfurter | Free, no key |
| Hotels & Attractions | Tavily Search | Free tier |
| Geocoding | Nominatim / OSM | Free, no key |

---
## 1 · Install Dependencies

In [24]:
!pip install -q langgraph langchain langchain-groq langchain-community \
    tavily-python python-dotenv pydantic requests geopy

---
## 2 · Imports & Configuration

In [25]:
import os, json, re, requests
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, Any, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# LangChain / LangGraph
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# Tavily search
from tavily import TavilyClient

# Geocoding
from geopy.geocoders import Nominatim

# ── Load environment variables ──────────────────────────────────────
load_dotenv()

GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found in .env")

# ── Initialize clients ──────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=GROQ_API_KEY,
)

tavily = TavilyClient(api_key=TAVILY_API_KEY)
geocoder = Nominatim(user_agent="ai-travel-agent-notebook")

print("✅ All clients initialized successfully.")

✅ All clients initialized successfully.


---
## 3 · Data Models & Agent State

In [26]:
class TravelRequest(BaseModel):
    """Structured representation of a user's travel request."""
    destination: str = Field(description="City or place to visit")
    num_days: int = Field(description="Number of days for the trip")
    budget: float = Field(description="Total budget amount")
    budget_currency: str = Field(description="Currency of the budget (ISO 4217)")
    home_currency: str = Field(description="User's home currency (ISO 4217)")
    interests: list[str] = Field(default_factory=list, description="List of interests / preferences")
    start_date: str = Field(default="", description="Trip start date (YYYY-MM-DD), empty if not specified")


class HotelOption(BaseModel):
    name: str
    price_per_night: str
    rating: str = ""
    source_url: str = ""
    snippet: str = ""


class Attraction(BaseModel):
    name: str
    category: str = ""
    description: str = ""
    source_url: str = ""


class WeatherDay(BaseModel):
    date: str
    temp_max_c: float
    temp_min_c: float
    precipitation_mm: float = 0.0
    description: str = ""


# ── Agent State ────────────────────────────────────────────────────
class AgentState(TypedDict):
    user_prompt: str
    is_travel: bool
    rejection_message: str
    travel_request: dict | None
    latitude: float
    longitude: float
    weather: list[dict]
    currency_rate: float
    budget_in_home: float
    hotels: list[dict]
    attractions: list[dict]
    budget_breakdown: dict
    itinerary_md: str
    summary_md: str
    error: str
    # ── Feedback loop fields ──
    calc_retries: int       # times calculator has re-run
    itin_retries: int       # times itinerary has re-run
    budget_valid: bool      # did itinerary pass budget check?
    itinerary_valid: bool   # did summary pass quality check?

MAX_RETRIES = 2  # prevent infinite loops
print("✅ Data models defined.")

✅ Data models defined.


---
## 4 · Tool Functions (Geocoding, Weather, Currency, Hotels, Attractions)

In [27]:
# ═══════════════════════════════════════════════════════════════════
#  GEOCODING
# ═══════════════════════════════════════════════════════════════════

def geocode_city(city_name: str) -> tuple[float, float]:
    """Convert a city name to (latitude, longitude) using Nominatim."""
    location = geocoder.geocode(city_name)
    if location is None:
        raise ValueError(f"Could not geocode '{city_name}'")
    return (location.latitude, location.longitude)


# ═══════════════════════════════════════════════════════════════════
#  WEATHER  (Open-Meteo — free, no key)
# ═══════════════════════════════════════════════════════════════════

WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def get_weather_forecast(lat: float, lon: float, start_date: str, num_days: int) -> list[dict]:
    """Fetch daily weather forecast from Open-Meteo."""
    sd = datetime.strptime(start_date, "%Y-%m-%d")
    ed = sd + timedelta(days=num_days - 1)

    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weather_code",
        "start_date": start_date, "end_date": ed.strftime("%Y-%m-%d"),
        "timezone": "auto",
    }, timeout=10)
    resp.raise_for_status()
    data = resp.json()["daily"]

    return [
        WeatherDay(
            date=data["time"][i],
            temp_max_c=data["temperature_2m_max"][i],
            temp_min_c=data["temperature_2m_min"][i],
            precipitation_mm=data["precipitation_sum"][i] or 0.0,
            description=WMO_CODES.get(data["weather_code"][i], f"Code {data['weather_code'][i]}"),
        ).model_dump()
        for i in range(len(data["time"]))
    ]


# ═══════════════════════════════════════════════════════════════════
#  CURRENCY  (Frankfurter API — free, no key)
# ═══════════════════════════════════════════════════════════════════

def get_exchange_rate(base: str, quote: str) -> float:
    """Fetch latest exchange rate from Frankfurter API."""
    base, quote = base.upper(), quote.upper()
    if base == quote:
        return 1.0
    resp = requests.get(f"https://api.frankfurter.dev/v2/rates?base={base}&quotes={quote}", timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, list) and len(data) > 0:
        return data[0]["rate"]
    elif isinstance(data, dict) and "rates" in data:
        return data["rates"][quote]
    raise ValueError(f"Unexpected Frankfurter response: {data}")


def convert_currency(amount: float, base: str, quote: str) -> tuple[float, float]:
    """Returns (rate, converted_amount)."""
    rate = get_exchange_rate(base, quote)
    return rate, round(amount * rate, 2)


# ═══════════════════════════════════════════════════════════════════
#  HOTEL SEARCH  (Tavily)
# ═══════════════════════════════════════════════════════════════════

def search_hotels(destination: str, budget_per_night: float, currency: str, num_days: int, start_date: str = "") -> list[dict]:
    """Search for hotels using Tavily."""
    date_hint = f" for {start_date}" if start_date else ""
    query = f"best hotels in {destination} under {budget_per_night:.0f} {currency} per night{date_hint} {num_days} nights, with reviews and prices"
    results = tavily.search(query=query, max_results=5, search_depth="advanced")

    snippets = "\n\n".join(
        f"Source: {r.get('url', '')}\nTitle: {r.get('title', '')}\nContent: {r.get('content', '')}"
        for r in results.get("results", [])
    )

    extract_prompt = f"""From the following search results about hotels in {destination}, extract up to 5 hotel options.
For each hotel provide:
- name (string)
- price_per_night (string, e.g. "~120 EUR")
- rating (string, e.g. "4.5/5" or "" if not available)
- source_url (string)
- snippet (string, one-line description)

ALL fields must be strings. Never use null. Use empty string "" if data is unavailable.
Return ONLY a valid JSON array. No markdown, no explanation.

Search results:
{snippets}"""

    response = llm.invoke([HumanMessage(content=extract_prompt)])
    try:
        text = re.sub(r"^```(?:json)?\s*", "", response.content.strip())
        text = re.sub(r"\s*```$", "", text)
        hotels = json.loads(text)
        return [HotelOption(**h).model_dump() for h in hotels[:5]]
    except Exception as e:
        print(f"⚠️  Hotel parsing fallback: {e}")
        return [
            HotelOption(name=r.get("title", "Unknown Hotel"), price_per_night="See link",
                        source_url=r.get("url", ""), snippet=r.get("content", "")[:200]).model_dump()
            for r in results.get("results", [])[:5]
        ]


# ═══════════════════════════════════════════════════════════════════
#  ATTRACTIONS SEARCH  (Tavily)
# ═══════════════════════════════════════════════════════════════════

def search_attractions(destination: str, interests: list[str], num_days: int) -> list[dict]:
    """Search for attractions matching user interests."""
    interest_str = ", ".join(interests) if interests else "popular sightseeing"
    query = f"top things to do and attractions in {destination} for someone interested in {interest_str}, best for a {num_days}-day trip"
    results = tavily.search(query=query, max_results=7, search_depth="advanced")

    snippets = "\n\n".join(
        f"Source: {r.get('url', '')}\nTitle: {r.get('title', '')}\nContent: {r.get('content', '')}"
        for r in results.get("results", [])
    )

    extract_prompt = f"""From the search results about attractions in {destination} (interests: {interest_str}), extract up to {num_days * 3} attractions.
For each provide:
- name (string)
- category (string, e.g. "Museum", "Landmark", "Park", "Food")
- description (string, 1-2 sentences)
- source_url (string)

ALL fields must be strings. Never use null.
Return ONLY a valid JSON array. No markdown.

Search results:
{snippets}"""

    response = llm.invoke([HumanMessage(content=extract_prompt)])
    try:
        text = re.sub(r"^```(?:json)?\s*", "", response.content.strip())
        text = re.sub(r"\s*```$", "", text)
        return [Attraction(**a).model_dump() for a in json.loads(text)]
    except Exception as e:
        print(f"⚠️  Attraction parsing fallback: {e}")
        return [
            Attraction(name=r.get("title", "Unknown"), description=r.get("content", "")[:200],
                       source_url=r.get("url", "")).model_dump()
            for r in results.get("results", [])[:num_days * 3]
        ]


print("✅ All tool functions defined.")

✅ All tool functions defined.


---
## 5 · LangGraph Agent Pipeline

**Features:**
- ✅ Conditional routing (TRAVEL / NOT_TRAVEL)
- ✅ Parallel data gathering (hotel, weather, attractions)
- ✅ Dedicated calculator, itinerary, and summary agents
- 🔄 **Feedback loop 1**: itinerary → calculator (if budget exceeded)
- 🔄 **Feedback loop 2**: summary → itinerary (if sections missing)
- 🛡️ Max 2 retries to prevent infinite loops

In [28]:
# ═══════════════════════════════════════════════════════════════════
#  NODE 1: TRAVEL / NOT_TRAVEL ROUTER
# ═══════════════════════════════════════════════════════════════════

def node_router(state: AgentState) -> dict:
    """Classify the query as TRAVEL or NOT_TRAVEL."""
    print("\n🔀 Routing query...")
    response = llm.invoke([
        SystemMessage(content=(
            "You are a query classifier. Determine if the user's message is a travel planning request.\n"
            "Reply with ONLY one word: TRAVEL or NOT_TRAVEL.\n"
            "TRAVEL = user wants to plan a trip, find hotels, visit places, travel itinerary, etc.\n"
            "NOT_TRAVEL = anything else (general questions, coding help, math, etc.)"
        )),
        HumanMessage(content=state["user_prompt"]),
    ])
    classification = response.content.strip().upper()
    is_travel = "TRAVEL" in classification and "NOT" not in classification
    if is_travel:
        print("   ✅ Classified as TRAVEL request")
        return {"is_travel": True, "rejection_message": ""}
    else:
        msg = "❌ This doesn't look like a travel request. Please describe a trip you'd like to plan!"
        print(f"   {msg}")
        return {"is_travel": False, "rejection_message": msg}


def route_condition(state: AgentState) -> Literal["query_analyzer", "__end__"]:
    return "query_analyzer" if state["is_travel"] else "__end__"


# ═══════════════════════════════════════════════════════════════════
#  NODE 2: QUERY ANALYZER
# ═══════════════════════════════════════════════════════════════════

def node_query_analyzer(state: AgentState) -> dict:
    """Parse travel request into structured data."""
    print("\n🧠 Analyzing your travel request...")
    today = datetime.now().strftime("%Y-%m-%d")
    tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
    response = llm.invoke([
        SystemMessage(content=f"""You are a travel-request parser. Today is {today}.
Extract these fields and return ONLY valid JSON (no markdown):
{{
  "destination": "city name",
  "num_days": integer,
  "budget": number,
  "budget_currency": "ISO 4217",
  "home_currency": "ISO 4217",
  "interests": ["interest1", "interest2"],
  "start_date": "YYYY-MM-DD"
}}
Defaults: start_date → {tomorrow}, home_currency → USD, budget_currency → USD."""),
        HumanMessage(content=state["user_prompt"]),
    ])
    text = re.sub(r"^```(?:json)?\s*", "", response.content.strip())
    text = re.sub(r"\s*```$", "", text)
    tr = TravelRequest(**json.loads(text))
    print(f"   📍 Destination : {tr.destination}")
    print(f"   📅 Days        : {tr.num_days}")
    print(f"   💰 Budget      : {tr.budget} {tr.budget_currency}")
    print(f"   🏠 Home currency: {tr.home_currency}")
    print(f"   🎯 Interests   : {', '.join(tr.interests)}")
    print(f"   📆 Start date  : {tr.start_date}")
    return {"travel_request": tr.model_dump()}


# ═══════════════════════════════════════════════════════════════════
#  NODE 3: GEOCODE
# ═══════════════════════════════════════════════════════════════════

def node_geocode(state: AgentState) -> dict:
    dest = state["travel_request"]["destination"]
    print(f"\n📍 Geocoding {dest}...")
    lat, lon = geocode_city(dest)
    print(f"   → lat={lat:.4f}, lon={lon:.4f}")
    return {"latitude": lat, "longitude": lon}


# ═══════════════════════════════════════════════════════════════════
#  NODES 4a/4b/4c: PARALLEL DATA GATHERING
# ═══════════════════════════════════════════════════════════════════

def node_hotel_agent(state: AgentState) -> dict:
    tr = state["travel_request"]
    nightly = tr["budget"] / tr["num_days"] * 0.5
    print(f"\n🏨 Hotel Agent: searching (≤{nightly:.0f} {tr['budget_currency']}/night)...")
    hotels = search_hotels(tr["destination"], nightly, tr["budget_currency"], tr["num_days"], tr["start_date"])
    for h in hotels[:3]:
        print(f"   • {h['name']} — {h['price_per_night']}")
    return {"hotels": hotels}


def node_weather_agent(state: AgentState) -> dict:
    tr = state["travel_request"]
    print(f"\n🌤  Weather Agent: fetching forecast for {tr['destination']}...")
    weather = get_weather_forecast(state["latitude"], state["longitude"], tr["start_date"], tr["num_days"])
    for w in weather:
        print(f"   {w['date']}: {w['temp_min_c']}–{w['temp_max_c']}°C, {w['description']}")
    return {"weather": weather}


def node_attractions_agent(state: AgentState) -> dict:
    tr = state["travel_request"]
    print(f"\n🎨 Attractions Agent: finding places in {tr['destination']}...")
    attractions = search_attractions(tr["destination"], tr["interests"], tr["num_days"])
    for a in attractions[:5]:
        print(f"   • {a['name']} ({a['category']})")
    return {"attractions": attractions}


# ═══════════════════════════════════════════════════════════════════
#  NODE 5: CALCULATOR AGENT (with retry awareness)
# ═══════════════════════════════════════════════════════════════════

def node_calculator_agent(state: AgentState) -> dict:
    """Handle currency conversion and budget breakdown. On retry, adjusts allocations."""
    tr = state["travel_request"]
    base, quote = tr["budget_currency"], tr["home_currency"]
    retries = state.get("calc_retries", 0)

    if retries > 0:
        print(f"\n🔄 Calculator Agent: RETRY #{retries} — adjusting budget allocation...")
    else:
        print(f"\n🧮 Calculator Agent: converting {tr['budget']} {base} → {quote}...")

    rate, total_home = convert_currency(tr["budget"], base, quote)
    budget = tr["budget"]

    # On retry, shift budget away from accommodation toward activities
    if retries > 0:
        accom_pct = max(0.30, 0.40 - retries * 0.05)
        food_pct = 0.25
        act_pct = min(0.25, 0.15 + retries * 0.05)
        trans_pct = 0.10
        misc_pct = round(1.0 - accom_pct - food_pct - act_pct - trans_pct, 2)
    else:
        accom_pct, food_pct, act_pct, trans_pct, misc_pct = 0.40, 0.25, 0.15, 0.10, 0.10

    breakdown = {
        "accommodation": round(budget * accom_pct, 2),
        "food": round(budget * food_pct, 2),
        "activities": round(budget * act_pct, 2),
        "transportation": round(budget * trans_pct, 2),
        "miscellaneous": round(budget * misc_pct, 2),
        "currency": base, "rate": rate,
        "total_budget": budget, "total_in_home": total_home, "home_currency": quote,
    }

    print(f"   Rate: 1 {base} = {rate} {quote}")
    print(f"   Total: {budget} {base} = {total_home} {quote}")
    print(f"   📊 Allocation: 🏨{breakdown['accommodation']} | 🍽️{breakdown['food']} | 🎟️{breakdown['activities']} | 🚕{breakdown['transportation']} | 🛍️{breakdown['miscellaneous']} {base}")

    return {"currency_rate": rate, "budget_in_home": total_home, "budget_breakdown": breakdown,
            "calc_retries": retries + 1}


# ═══════════════════════════════════════════════════════════════════
#  NODE 6: ITINERARY AGENT (with budget validation)
# ═══════════════════════════════════════════════════════════════════

def node_itinerary_agent(state: AgentState) -> dict:
    """Compose itinerary and validate it against the budget."""
    tr = state["travel_request"]
    bb = state["budget_breakdown"]
    retries = state.get("itin_retries", 0)

    if retries > 0:
        print(f"\n🔄 Itinerary Agent: RETRY #{retries} — refining plan to fit budget...")
    else:
        print("\n📝 Itinerary Agent: composing day-by-day plan...")

    budget_warning = ""
    if retries > 0:
        budget_warning = f"""\n\n⚠️ IMPORTANT: The previous itinerary exceeded the budget.
You MUST keep the total under {tr["budget"]} {tr["budget_currency"]}.
Use cheaper alternatives, fewer paid activities, or free options."""

    prompt = f"""You are an expert travel planner. Create a detailed DAY-BY-DAY ITINERARY.

## Trip Details
- Destination: {tr["destination"]}
- Duration: {tr["num_days"]} days (starting {tr["start_date"]})
- Budget: {tr["budget"]} {tr["budget_currency"]} (≈ {state["budget_in_home"]} {tr["home_currency"]})
- Interests: {', '.join(tr['interests'])}

## Budget Allocation
- Accommodation: {bb["accommodation"]} {bb["currency"]}/trip
- Food: {bb["food"]} {bb["currency"]}/trip
- Activities: {bb["activities"]} {bb["currency"]}/trip
- Transport: {bb["transportation"]} {bb["currency"]}/trip

## Weather
{json.dumps(state["weather"], indent=2)}

## Hotels
{json.dumps(state["hotels"], indent=2)}

## Attractions
{json.dumps(state["attractions"], indent=2)}
{budget_warning}

## Instructions
For EACH day: date, weather, Morning/Afternoon/Evening activities, meals, estimated daily spend.
Pick the BEST hotel. Use emojis. Output Markdown.
At the END, add a line: "TOTAL_ESTIMATED: <number> <currency>" with the grand total.
The total MUST be ≤ {tr["budget"]} {tr["budget_currency"]}."""

    response = llm.invoke([HumanMessage(content=prompt)])
    itinerary = response.content.strip()

    # ── Budget validation: check if total exceeds budget ──
    budget_valid = True
    import re as _re
    match = _re.search(r"TOTAL_ESTIMATED:\s*([\d,.]+)", itinerary)
    if match:
        try:
            estimated = float(match.group(1).replace(",", ""))
            if estimated > tr["budget"] * 1.1:  # 10% tolerance
                budget_valid = False
                print(f"   ⚠️ Budget exceeded! Estimated {estimated} vs budget {tr['budget']}")
        except ValueError:
            pass

    if budget_valid:
        print("   ✅ Itinerary composed & budget validated!")
    else:
        print(f"   🔄 Budget validation FAILED (retry {retries + 1}/{MAX_RETRIES})")

    return {"itinerary_md": itinerary, "budget_valid": budget_valid,
            "itin_retries": retries + 1}


# ── Conditional: itinerary → calculator (retry) or summary (proceed) ──
def route_after_itinerary(state: AgentState) -> Literal["calculator_agent", "summary_agent"]:
    """If budget invalid and retries left, loop back to calculator."""
    if not state.get("budget_valid", True) and state.get("itin_retries", 0) <= MAX_RETRIES:
        return "calculator_agent"
    return "summary_agent"


# ═══════════════════════════════════════════════════════════════════
#  NODE 7: SUMMARY AGENT (with quality validation)
# ═══════════════════════════════════════════════════════════════════

def node_summary_agent(state: AgentState) -> dict:
    """Create final summary. Validates quality and can retry itinerary."""
    tr = state["travel_request"]
    bb = state["budget_breakdown"]
    print("\n📋 Summary Agent: finalizing trip plan...")

    prompt = f"""You are a travel document formatter. Combine the itinerary into a polished plan.

## Itinerary
{state["itinerary_md"]}

## Budget Data
- Total: {bb["total_budget"]} {bb["currency"]} = {bb["total_in_home"]} {bb["home_currency"]}
- Rate: 1 {bb["currency"]} = {bb["rate"]} {bb["home_currency"]}
- Accommodation: {bb["accommodation"]} {bb["currency"]}
- Food: {bb["food"]} {bb["currency"]}
- Activities: {bb["activities"]} {bb["currency"]}
- Transportation: {bb["transportation"]} {bb["currency"]}
- Miscellaneous: {bb["miscellaneous"]} {bb["currency"]}

## Weather
{json.dumps(state["weather"], indent=2)}

## Instructions
Create a polished Markdown document with:
1. 🌍 Trip Overview (destination, dates, budget both currencies)
2. 🏨 Recommended Hotel (price, reason)
3. 📅 Day-by-Day Itinerary
4. 💰 Budget Breakdown (table, BOTH currencies)
5. 🧳 Travel Tips (weather-based packing, customs, phrases)

Use emojis. All costs in BOTH currencies."""

    response = llm.invoke([HumanMessage(content=prompt)])
    summary = response.content.strip()

    # ── Quality validation: check key sections exist ──
    required_sections = ["Hotel", "Day", "Budget", "Tips"]
    missing = [s for s in required_sections if s.lower() not in summary.lower()]
    itinerary_valid = len(missing) == 0

    if not itinerary_valid and state.get("itin_retries", 0) <= MAX_RETRIES:
        print(f"   ⚠️ Missing sections: {missing} — will retry itinerary")
    else:
        # Save to file
        dest = tr["destination"].replace(" ", "_")
        filepath = f"trip_plan_{dest}.md"
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(summary)
        print(f"   ✅ Trip plan saved to: {filepath}")
        itinerary_valid = True  # force accept if out of retries

    return {"summary_md": summary, "itinerary_valid": itinerary_valid}


# ── Conditional: summary → itinerary (retry) or END (done) ──
def route_after_summary(state: AgentState) -> Literal["itinerary_agent", "__end__"]:
    """If summary quality check failed and retries left, redo itinerary."""
    if not state.get("itinerary_valid", True) and state.get("itin_retries", 0) <= MAX_RETRIES:
        return "itinerary_agent"
    return "__end__"


# ═══════════════════════════════════════════════════════════════════
#  BUILD THE GRAPH (with feedback loops)
# ═══════════════════════════════════════════════════════════════════

def build_travel_agent():
    graph = StateGraph(AgentState)

    # Add all nodes
    graph.add_node("router", node_router)
    graph.add_node("query_analyzer", node_query_analyzer)
    graph.add_node("geocode", node_geocode)
    graph.add_node("hotel_agent", node_hotel_agent)
    graph.add_node("weather_agent", node_weather_agent)
    graph.add_node("attractions_agent", node_attractions_agent)
    graph.add_node("calculator_agent", node_calculator_agent)
    graph.add_node("itinerary_agent", node_itinerary_agent)
    graph.add_node("summary_agent", node_summary_agent)

    # START → router
    graph.add_edge(START, "router")

    # router → TRAVEL or NOT_TRAVEL
    graph.add_conditional_edges("router", route_condition)

    # query_analyzer → geocode
    graph.add_edge("query_analyzer", "geocode")

    # Parallel fan-out: geocode → [hotel, weather, attractions]
    graph.add_edge("geocode", "hotel_agent")
    graph.add_edge("geocode", "weather_agent")
    graph.add_edge("geocode", "attractions_agent")

    # Fan-in → calculator
    graph.add_edge("hotel_agent", "calculator_agent")
    graph.add_edge("weather_agent", "calculator_agent")
    graph.add_edge("attractions_agent", "calculator_agent")

    # calculator → itinerary (always)
    graph.add_edge("calculator_agent", "itinerary_agent")

    # ── FEEDBACK LOOP 1: itinerary → calculator (budget retry) or summary ──
    graph.add_conditional_edges("itinerary_agent", route_after_itinerary)

    # ── FEEDBACK LOOP 2: summary → itinerary (quality retry) or END ──
    graph.add_conditional_edges("summary_agent", route_after_summary)

    return graph.compile()


agent = build_travel_agent()
print("✅ Travel agent graph compiled (with feedback loops)!")
print("\n📊 Graph with dotted feedback edges:")
print("   START → router → [TRAVEL] → query_analyzer → geocode")
print("                                                    ↓")
print("                                 ┌─────────────────┼─────────────────┐")
print("                                 ▼                 ▼                 ▼")
print("                           hotel_agent      weather_agent     attractions_agent")
print("                                 └─────────────────┼─────────────────┘")
print("                                                   ▼")
print("                                  ┌──────── calculator_agent ◄────────┐")
print("                                  │                ▼                  │")
print("                                  │        itinerary_agent ───────────┘ (budget retry)")
print("                                  │                ▼")
print("                       (quality   └─────── summary_agent")
print("                        retry)                    ▼")
print("                                                 END")

✅ Travel agent graph compiled (with feedback loops)!

📊 Graph with dotted feedback edges:
   START → router → [TRAVEL] → query_analyzer → geocode
                                                    ↓
                                 ┌─────────────────┼─────────────────┐
                                 ▼                 ▼                 ▼
                           hotel_agent      weather_agent     attractions_agent
                                 └─────────────────┼─────────────────┘
                                                   ▼
                                  ┌──────── calculator_agent ◄────────┐
                                  │                ▼                  │
                                  │        itinerary_agent ───────────┘ (budget retry)
                                  │                ▼
                       (quality   └─────── summary_agent
                        retry)                    ▼
                                                 END


---
## 6 · Run the Agent

In [29]:
def plan_trip(user_prompt: str) -> str:
    """
    Run the AI travel agent on a natural-language prompt.
    Returns the Markdown trip plan (also saves to file).
    """
    print("═" * 60)
    print("  🌍  AI TRAVEL AGENT  ")
    print("═" * 60)
    print(f'\n📩 Your request:\n   "{user_prompt}"')

    initial_state: AgentState = {
        "user_prompt": user_prompt,
        "is_travel": False,
        "rejection_message": "",
        "travel_request": None,
        "latitude": 0.0,
        "longitude": 0.0,
        "weather": [],
        "currency_rate": 0.0,
        "budget_in_home": 0.0,
        "hotels": [],
        "attractions": [],
        "budget_breakdown": {},
        "itinerary_md": "",
        "summary_md": "",
        "error": "",
        # Feedback loop init
        "calc_retries": 0,
        "itin_retries": 0,
        "budget_valid": True,
        "itinerary_valid": True,
    }

    result = agent.invoke(initial_state)

    if not result["is_travel"]:
        print(f'\n{result["rejection_message"]}')
        return result["rejection_message"]

    retries_used = result.get("calc_retries", 1) - 1 + result.get("itin_retries", 1) - 1
    if retries_used > 0:
        print(f"\n🔄 Self-corrected {retries_used} time(s) via feedback loops")

    print(f'\n{"═" * 60}')
    print("  ✅ TRIP PLAN COMPLETE!")
    print(f'{"═" * 60}')
    return result["summary_md"]

print("✅ plan_trip() ready to use.")

✅ plan_trip() ready to use.


---
## 7 · 🚀 Demo — Plan a Trip!

Try changing the prompt below to plan your own trip:

In [30]:
# ── Your travel request ────────────────────────────────────────────
user_request = (
    "Plan me a 3-day trip to Paris, "
    "my budget is 1000 EUR, "
    "I like art museums and French cuisine, "
    "and my home currency is USD."
)

# ── Run the agent ──────────────────────────────────────────────────
itinerary = plan_trip(user_request)

════════════════════════════════════════════════════════════
  🌍  AI TRAVEL AGENT  
════════════════════════════════════════════════════════════

📩 Your request:
   "Plan me a 3-day trip to Paris, my budget is 1000 EUR, I like art museums and French cuisine, and my home currency is USD."

🔀 Routing query...
   ✅ Classified as TRAVEL request

🧠 Analyzing your travel request...
   📍 Destination : Paris
   📅 Days        : 3
   💰 Budget      : 1000.0 EUR
   🏠 Home currency: USD
   🎯 Interests   : art museums, French cuisine
   📆 Start date  : 2026-04-11

📍 Geocoding Paris...
   → lat=48.8535, lon=2.3484

🎨 Attractions Agent: finding places in Paris...

🏨 Hotel Agent: searching (≤167 EUR/night)...

🌤  Weather Agent: fetching forecast for Paris...
   2026-04-11: 9.4–14.6°C, Moderate rain
   2026-04-12: 6.4–15.3°C, Overcast
   2026-04-13: 7.9–15.3°C, Slight rain
   • Hotel Ampere — ~170 EUR
   • Hotel Grands Boulevards — ~200 EUR
   • Hotel Mathis Paris — 
   • The Louvre (Museum)
   • Musée 

### 📄 View the Generated Plan

In [31]:
from IPython.display import Markdown, display
display(Markdown(itinerary))

🌍 Trip Overview
===============

Destination: Paris, France 🇫🇷
Dates: 2026-04-11 to 2026-04-13
Budget: 1000.0 EUR (1168.2 USD)

🏨 Recommended Hotel
=====================

The recommended hotel is **Hotel Ampere** 🏨, with a price of ~170 EUR (~198.3 USD) per night. For 3 nights, the total cost would be ~510 EUR (~595.0 USD). This hotel is chosen for its affordability and proximity to major attractions.

📅 Day-by-Day Itinerary
=======================

### Day 1: 2026-04-11 🌂
- **Weather**: Moderate rain ☔️ (14.6°C - 9.4°C)
- **Morning**: Visit the Louvre Museum 🎨 (9:00 AM - 12:00 PM, 18 EUR / 21.0 USD)
- **Afternoon**: Explore the Strada Cafe for coffee and people-watching 🍵 (1:00 PM - 3:00 PM, 10 EUR / 11.7 USD)
- **Evening**: Enjoy dinner at a traditional French bistro 🍴 (6:00 PM - 8:00 PM, 25 EUR / 29.2 USD)
- **Meals**: Breakfast at the hotel 🥐 (10 EUR / 11.7 USD), Lunch at a street food stall 🍔 (15 EUR / 17.5 USD), Dinner at the bistro 🍴 (25 EUR / 29.2 USD)
- **Estimated Daily Spend**: 78 EUR / 91.2 USD

### Day 2: 2026-04-12 ⛅️
- **Weather**: Overcast ☁️ (15.3°C - 6.4°C)
- **Morning**: Visit the Musée d'Orsay 🖼️ (9:00 AM - 12:00 PM, 12 EUR / 14.0 USD)
- **Afternoon**: Wander around the Place du Tertre and enjoy the street artists 🎨 (1:00 PM - 4:00 PM, free)
- **Evening**: Relax with a mint tea at La Mosquee 🍵 (5:00 PM - 7:00 PM, 10 EUR / 11.7 USD)
- **Meals**: Breakfast at a local bakery 🥐 (12 EUR / 14.0 USD), Lunch at a café 🥗 (20 EUR / 23.4 USD), Dinner at La Mosquee 🍴 (30 EUR / 35.0 USD)
- **Estimated Daily Spend**: 84 EUR / 98.2 USD

### Day 3: 2026-04-13 🌂
- **Weather**: Slight rain 🌂 (15.3°C - 7.9°C)
- **Morning**: Explore the Musée de l'Orangerie 🌺 (9:00 AM - 12:00 PM, 10 EUR / 11.7 USD)
- **Afternoon**: Visit the Quai Branly Museum or the Picasso Museum 🎨 (1:00 PM - 4:00 PM, 12 EUR / 14.0 USD)
- **Evening**: Farewell dinner at a Michelin-starred restaurant 🍴 (6:00 PM - 9:00 PM, 40 EUR / 46.7 USD)
- **Meals**: Breakfast at the hotel 🥐 (10 EUR / 11.7 USD), Lunch at a food truck 🌮 (15 EUR / 17.5 USD), Dinner at the Michelin-starred restaurant 🍴 (40 EUR / 46.7 USD)
- **Estimated Daily Spend**: 87 EUR / 101.7 USD

💰 Budget Breakdown
=====================

| Category | EUR | USD |
| --- | --- | --- |
| Accommodation | 510.0 | 595.0 |
| Food | 250.0 | 292.0 |
| Activities | 150.0 | 175.0 |
| Transportation | 100.0 | 117.0 |
| Miscellaneous | 100.0 | 117.0 |
| **Total** | **1010.0** | **1174.0** |

However, the total estimated spend exceeds the budget. To adjust, consider reducing food expenses by choosing more affordable options or cutting back on one of the museum visits. 

Revised **TOTAL_ESTIMATED: 970.0 EUR** (~1135.0 USD) after adjusting food expenses and considering free museum visits on the first Sunday of the month for some museums.

🧳 Travel Tips
==============

* Pack accordingly for the weather: ☔️ Moderate rain, ☁️ Overcast, and 🌂 Slight rain. Bring an umbrella and waterproof shoes.
* Learn basic French phrases: Bonjour (hello), Merci (thank you), Au revoir (goodbye).
* Respect local customs: remove your hat when entering a church or museum, and avoid eating on the go.
* Don't forget to try some delicious French cuisine and pastries! 🥐🍽️

Note: The exchange rate used is 1 EUR = 1.1682 USD.

### 🚫 Test: Non-Travel Query (should be rejected)

In [32]:
# This should be classified as NOT_TRAVEL and rejected gracefully
rejection = plan_trip("What is the capital of France?")

════════════════════════════════════════════════════════════
  🌍  AI TRAVEL AGENT  
════════════════════════════════════════════════════════════

📩 Your request:
   "What is the capital of France?"

🔀 Routing query...
   ❌ This doesn't look like a travel request. Please describe a trip you'd like to plan!

❌ This doesn't look like a travel request. Please describe a trip you'd like to plan!


---
### 🔄 Try Another Trip

In [33]:
custom_request = (
    "I want a 5-day trip to Tokyo, Japan. "
    "My budget is 2000 USD. "
    "I love street food, anime culture, and temples. "
    "Home currency is INR."
)

custom_itinerary = plan_trip(custom_request)
display(Markdown(custom_itinerary))

════════════════════════════════════════════════════════════
  🌍  AI TRAVEL AGENT  
════════════════════════════════════════════════════════════

📩 Your request:
   "I want a 5-day trip to Tokyo, Japan. My budget is 2000 USD. I love street food, anime culture, and temples. Home currency is INR."

🔀 Routing query...
   ✅ Classified as TRAVEL request

🧠 Analyzing your travel request...
   📍 Destination : Tokyo
   📅 Days        : 5
   💰 Budget      : 2000.0 USD
   🏠 Home currency: INR
   🎯 Interests   : street food, anime culture, temples
   📆 Start date  : 2026-04-11

📍 Geocoding Tokyo...
   → lat=35.6769, lon=139.7639

🎨 Attractions Agent: finding places in Tokyo...

🏨 Hotel Agent: searching (≤200 USD/night)...

🌤  Weather Agent: fetching forecast for Tokyo...
   2026-04-11: 13.2–24.6°C, Overcast
   2026-04-12: 10.9–22.0°C, Overcast
   2026-04-13: 9.1–23.2°C, Overcast
   2026-04-14: 15.4–20.4°C, Slight rain
   2026-04-15: 11.5–18.5°C, Slight rain
   • HOTEL TAVINOS Asakusa — ~72 SGD
   

🌍 Trip Overview
===============

Destination: Tokyo, Japan
Dates: 2026-04-11 to 2026-04-15
Budget: $1400.0 USD (₹129,082.0 INR) out of $2000.0 USD (₹185,260.0 INR)

🏨 Recommended Hotel
=====================

We recommend staying at **HOTEL TAVINOS Asakusa** 🏨, a chic design hotel near Senso-ji, with a price per night of ~$50 USD (₹4,631.5 INR). This hotel is ideal for its proximity to major attractions and its modern amenities.

📅 Day-by-Day Itinerary
=======================

### Day 1: 2026-04-11
🌫️ Overcast, High: 24.6°C, Low: 13.2°C
* Morning: Visit **Senso-ji** 🏯, Tokyo's oldest and most significant Buddhist temple, located in Asakusa.
* Afternoon: Explore **Takeshita-Dori** 🍴, a famous street in Harajuku lined with rainbow-colored, 'kawaii' foods.
* Evening: Walk around **Shimokitazawa** 🌃, a trendy neighborhood with vintage clothes and record stores, funky cafes, and bars with live music.
* Meals: Try some street food at Takeshita-Dori for lunch 🍔 and dinner at a local restaurant in Shimokitazawa 🍜.
* Estimated daily spend: $150 USD (₹13,844.5 INR)

### Day 2: 2026-04-12
🌫️ Overcast, High: 22.0°C, Low: 10.9°C
* Morning: Visit **Mandarake Complex** 📚, an eight-story building that specializes in anime, manga, and collectible toys.
* Afternoon: Explore **Akihabara** 🤖, a vibrant district in Tokyo known for its electronics shops, anime and manga stores, and pop culture attractions.
* Evening: Enjoy a viral cafe experience at **BONGENCOFFEE** ☕️, known for its signature matcha espresso.
* Meals: Grab lunch at a local cafe in Akihabara 🍞 and dinner at a restaurant in the Mandarake Complex 🍴.
* Estimated daily spend: $120 USD (₹11,135.6 INR)

### Day 3: 2026-04-13
🌫️ Overcast, High: 23.2°C, Low: 9.1°C
* Morning: Visit **Tsukiji Market** 🐟, a famous fish market in Tokyo where you can find fresh sashimi bowls and dishes.
* Afternoon: Explore **Ueno Park** 🌳, a lovely space to wander, with temples, food stalls, and the beautiful Shinobazu pond.
* Evening: Relax at the hotel or take a stroll around the nearby area 🌃.
* Meals: Try some fresh sushi at Tsukiji Market for lunch 🍣 and dinner at a local restaurant near Ueno Park 🍜.
* Estimated daily spend: $140 USD (₹12,948.2 INR)

### Day 4: 2026-04-14
⛅️ Slight rain, High: 20.4°C, Low: 15.4°C
* Morning: Visit **Kappabashi Street** 🍳, a kitchenware district where you can find Japanese kitchen knives and beautiful ceramic ware.
* Afternoon: Explore **Ginza** 🛍️, Tokyo's luxury shopping district, featuring flagship stores and high-end restaurants.
* Evening: Enjoy a high-end sushi experience at **Standing Sushi Hiroya** 🍣.
* Meals: Grab lunch at a local restaurant in Ginza 🍴 and dinner at Standing Sushi Hiroya 🍣.
* Estimated daily spend: $180 USD (₹16,673.4 INR)

### Day 5: 2026-04-15
⛅️ Slight rain, High: 18.5°C, Low: 11.5°C
* Morning: Visit **Gotokuji Temple** 🏯, a temple located in Shimokitazawa, known for its beautiful gardens and traditional architecture.
* Afternoon: Explore **Shinjuku Gyoen Garden** 🌳, a national garden with beautiful cherry blossom trees.
* Evening: Enjoy a farewell dinner at **Nabezo Shibuya** 🍜, an all-you-can-eat hot pot and sukiyaki spot in Shibuya.
* Meals: Try some local street food for lunch 🍔 and dinner at Nabezo Shibuya 🍜.
* Estimated daily spend: $160 USD (₹14,820.8 INR)

💰 Budget Breakdown
==================

| Category | Budget (USD) | Budget (INR) |
| --- | --- | --- |
| Accommodation | $400.0 | ₹36,952.0 |
| Food | $500.0 | ₹46,315.0 |
| Activities | $300.0 | ₹27,789.0 |
| Transportation | $200.0 | ₹18,526.0 |
| Miscellaneous | $200.0 | ₹18,526.0 |
| **Total** | **$1400.0** | **₹129,082.0** |

🧳 Travel Tips
==============

* Weather: Tokyo's weather will be overcast with slight rain on some days, so pack accordingly with waterproof gear and layers for temperature changes.
* Customs: Respect Japanese customs by bowing upon greeting, using chopsticks correctly, and removing shoes when required.
* Phrases:
	+ Hello: こんにちは (Konnichiwa)
	+ Thank you: ありがとう (Arigatou)
	+ Excuse me: すみません (Sumimasen)
* Packing:
	+ Comfortable shoes for walking
	+ Waterproof jacket or umbrella
	+ Power adapter for charging electronic devices
	+ Travel-sized essentials (toiletries, etc.)